# sweets demo

End-to-end walk-through of a small InSAR workflow with `sweets`. The same `Workflow` object handles three interchangeable input sources:

| `--source` | what it is |
|---|---|
| `safe` *(default)* | Raw Sentinel-1 bursts via `burst2safe` + COMPASS geocoding. |
| `opera-cslc` | Pre-made OPERA L2 CSLC-S1 HDF5s from ASF DAAC. Skips COMPASS. |
| `nisar-gslc` | Pre-made NISAR L2 GSLC HDF5s via CMR (L-band, UTM). Skips COMPASS and geometry stitching. |

Outputs land under `<work_dir>/dolphin/` in the dolphin output layout: `interferograms/`, `unwrapped/`, `timeseries/`, with `velocity.tif` as the top-level product.

## Environment setup

`sweets` is pixi-first. The `[tool.pixi.*]` sections in `pyproject.toml` are the canonical env definition and pin the forks of `s1-reader`, `COMPASS`, `opera-utils` and `dolphin` that carry the fixes sweets relies on.

```bash
git clone https://github.com/isce-framework/sweets.git && cd sweets
pixi install
pixi shell
```

## CLI surface

Three subcommands. `config` writes a YAML; `run` executes it.

In [ ]:
!sweets --help

In [ ]:
!sweets config --help

## Example 1: Sentinel-1 raw bursts (default, `--source safe`)

Tiny pecos AOI + a 2-week window. `--track` is required on this path. `--swaths IW2` keeps burst2safe on a single subswath (it refuses to straddle swaths).

```bash
sweets config \
  --bbox=-102.96 31.22 -101.91 31.56 \
  --start 2021-06-05 --end 2021-06-22 \
  --track 78 \
  --swaths IW2 \
  --out-dir ./data \
  --work-dir ./pecos_demo \
  --output pecos_demo/sweets_config.yaml

sweets run pecos_demo/sweets_config.yaml
```

## Example 2: OPERA CSLC (`--source opera-cslc`)

Same AOI but skipping burst2safe and COMPASS entirely. sweets pulls pre-geocoded OPERA CSLC HDF5s plus their CSLC-STATIC geometry layers directly from ASF. `--track` is optional — ASF filters on the AOI alone. Add `--do-tropo` to run the OPERA L4 TROPO-ZENITH correction step after dolphin.

```bash
sweets config \
  --bbox=-102.96 31.22 -101.91 31.56 \
  --start 2021-06-05 --end 2021-06-22 \
  --source opera-cslc \
  --do-tropo \
  --out-dir ./data \
  --work-dir ./pecos_opera \
  --output pecos_opera/sweets_config.yaml

sweets run pecos_opera/sweets_config.yaml
```

With `--do-tropo`, extra outputs show up under `dolphin/tropo/` (per-acquisition corrections) and `dolphin/tropo_corrected/<pair>.tropo_corrected.unw.tif`.

## Example 3: NISAR GSLC (`--source nisar-gslc`)

L-band, already geocoded. Pass `--track` and `--frame` to pin a single repeat-pass stack (same as the ASF Vertex filters). sweets peeks the HDF5 grid to pick a coherent (frequency, polarization) signature across cycles, writes tiny VRT wrappers that inject the real geotransform on top of the HDF5 subdataset, and hands the VRTs to dolphin. No COMPASS, no geometry stitching.

```bash
sweets config \
  --bbox=-121.10 36.55 -120.95 36.70 \
  --start 2025-10-01 --end 2025-12-15 \
  --source nisar-gslc \
  --track 42 --frame 70 \
  --out-dir ./data \
  --work-dir ./salinas_nisar \
  --output salinas_nisar/nisar.yaml

sweets run salinas_nisar/nisar.yaml
```

Tropo correction is not yet wired for NISAR (no stitched incidence-angle raster) — `--do-tropo` will warn and skip that step.

## Configuring from Python

The CLI just builds a `Workflow` and dumps it to YAML. You can do the same directly:

In [ ]:
from sweets.core import Workflow
from sweets.download import BurstSearch  # or OperaCslcSearch, NisarGslcSearch

w = Workflow(
    bbox=(-102.96, 31.22, -101.91, 31.56),
    search=BurstSearch(
        track=78,
        start="2021-06-05",
        end="2021-06-22",
        swaths=["IW2"],
        out_dir="data",
    ),
    work_dir="pecos_demo",
)

# Save for later / inspection / tweaking:
w.to_yaml("pecos_demo/sweets_config.yaml")

# Or run it now:
# w.run()

`Workflow.search` is a Pydantic discriminated union (`kind` field) — YAML files like `search: {kind: opera-cslc, ...}` load back into the matching variant via `Workflow.from_yaml`:

In [ ]:
w2 = Workflow.from_yaml("pecos_demo/sweets_config.yaml")
print(type(w2.search).__name__)  # BurstSearch, OperaCslcSearch, or NisarGslcSearch
# w2.run()

## Output layout

After `sweets run` finishes, the dolphin outputs live under `<work_dir>/dolphin/`:

```
dolphin/
  interferograms/<date1>_<date2>.int.tif   # wrapped ifgs + .cor.tif + .mask.tif
  unwrapped/<date1>_<date2>.unw.tif        # unwrapped phase + .unw.conncomp.tif
  timeseries/
    <date1>_<date2>.tif                    # per-date cumulative displacement (m)
    velocity.tif                           # phase velocity (m/yr)
    reference_point.txt
    conncomp_intersection.tif
  phase_linking/, PS/, ...                 # intermediate products
```

Timeseries and velocity rasters are in meters (radar wavelength is auto-detected from the source). Inspect with `gdalinfo` or open with `rasterio` / `xarray`.

For interactive browsing, load them straight into your preferred viewer (QGIS, matplotlib, `rioxarray` + `hvplot`, etc.) — the old `sweets.plotting.browse_ifgs` helper has been removed in favor of just reading the GeoTIFFs with standard tooling.